# scSVC reconstructs whole-genome profiles of Xenium datasets and defines fine-grained subtypes of T cells

In [ ]:
save_adata_flag = False

output_dir = "../../output/sc_SVC_case/"
select_ct = "T"

alpha = 0.2

In [ ]:
patient_id = "P2CRC"
data_type = "Xenium"
cell_type_col = "Level1"

raw_data_path = "../../raw_data/Real_application"
raw_file_name = f"{raw_data_path}/{patient_id}_{data_type}.h5ad"
sc_ref_file = f"{raw_data_path}/adata_sc_all_reanno.h5ad"

output_dir = f"{output_dir}/{patient_id}_{data_type}"

In [ ]:
import os

output_dir = f"{output_dir}/{select_ct}"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
import scanpy as sc
import pandas as pd

adata_sp = sc.read(raw_file_name)
# adata_sp.obs['total_counts'] = adata_sp.X.sum(axis = 1)
adata_sp = adata_sp[adata_sp.obs['transcript_counts']>=60,:]
sc.pp.filter_genes(adata_sp, min_cells=100)

# adata_sp.obs = adata_sp.obs[['cell_id','x','y']]
# adata_sp.obs_names = adata_sp.obs['cell_id']

adata_sc = sc.read(sc_ref_file)
adata_sc = adata_sc[adata_sc.obs['Patient'] == patient_id, :]
adata_sc.obs = adata_sc.obs[['Level1','Level2']]
sc.pp.filter_genes(adata_sc, min_cells=100)
adata_sc.obs['Level1'].replace({"Mono/Macro": "Mono_Macro"}, inplace=True)
adata_sc_raw = adata_sc.copy()

overlap_genes = adata_sp.var_names.intersection(adata_sc.var_names)
adata_sp = adata_sp[:, overlap_genes]

adata_sp

In [ ]:
from revise.methods.global_anchoring import GlobalAnchoring
from revise.application import ScSVC
from revise.tools.log import Logger
from revise.conf.application_sc_conf import ApplicationScConf

config = ApplicationScConf(
    sample_name=patient_id,
    raw_data_path=raw_data_path,
    result_root_path=output_dir,
    cell_type_col=cell_type_col,
    confidence_col="Confidence",
    unknown_key="Unknown",
    st_file=f"{data_type}.h5ad",
    sc_ref_file=sc_ref_file,
    # annotate_mode = 'pot'
)

logger = Logger(name=f"run.log", log_file=f'{output_dir}/application_sc.log').get_logger()
sc_svc = ScSVC(adata_sp, adata_sc, config, logger)
annotate_method = GlobalAnchoring(config, logger)
adata_sp = annotate_method.run(sc_svc.st_adata, sc_svc.sc_ref_adata, cell_type_col = "Level1")


In [ ]:
print(select_ct)
ct_adata_sc = adata_sc[adata_sc.obs['Level1'] == select_ct]
ct_adata_sp = adata_sp[adata_sp.obs['Level1'] == select_ct]
print(ct_adata_sc.n_obs, ct_adata_sp.n_obs)

In [ ]:
subcell_type_col = "Level2"
ct_adata_sp = annotate_method.run(ct_adata_sp, ct_adata_sc, cell_type_col = subcell_type_col)

In [ ]:
from revise.methods.graph_cluster import GraphCluster

resolutions = [0.6, 0.7, 0.8]

graph_cluster = GraphCluster(config, logger)
sc_SVC_adata, merge_df, best_res = graph_cluster.run(ct_adata_sp, resolutions, subcell_type_col)

sc_SVC_adata

In [ ]:
sc_SVC_adata.obs['SVC_cluster'] = sc_SVC_adata.obs[f'leiden_{best_res}']
sp_cluster_num = merge_df.loc[merge_df['resolution'] == best_res, 'cluster_num'].values[0]
sp_cluster_num

In [ ]:
ct_adata_sc = annotate_method.run(ct_adata_sc, sc_SVC_adata, cell_type_col = "SVC_cluster")

In [ ]:
if save_adata_flag:
    sc_SVC_adata.write(f"{output_dir}/sc_SVC_spatial.h5ad")
    ct_adata_sc.write(f"{output_dir}/sc_SVC_expr.h5ad")

print("Finish")